# Super-FunSearch for Online Bin Packing

This Colab notebook reproduces the main experimental result in the project report and documents the full Super-FunSearch pipeline.

## Project Summary

**Task.** We study one-dimensional online bin packing. Items arrive sequentially and must be placed immediately into a feasible bin. The goal is to minimize the final number of used bins.

**Heuristic interface.** Super-FunSearch evolves a Python priority function:

```python
priority(item: float, bins: np.ndarray) -> np.ndarray
```

`bins` contains the remaining capacities of feasible bins. The evaluator places the current item into the bin with the maximum returned priority score.

**What this notebook reproduces.** Without any LLM API key, it reproduces the report table for First Fit, Best Fit, Worst Fit, Next Fit, offline FFD, and our saved best Super-FunSearch heuristic on Weibull 5k.

## 1. Setup and Load Project Code

This cell locates the submitted `super_funsearch` code. In Colab, either set `REPO_URL` to your GitHub repository or upload this folder so that the notebook can find `super_funsearch/bench_heuristic.py` and `super_funsearch/bin_packing_utils.py`.

The deterministic reproduction path only needs `numpy` and `pandas`. The full optional LLM search may need the packages in `requirements.txt`.

In [ ]:
# Set this before submission if the code is hosted on GitHub.
# Example: REPO_URL = "https://github.com/<user>/<repo>.git"
REPO_URL = ""

import os
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path("/content/super-funsearch-submission")

if REPO_URL and not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)

# Candidate locations support both GitHub clone and manual zip upload.
candidate_dirs = [
    PROJECT_ROOT / "submission" / "super_funsearch_colab" / "super_funsearch",
    PROJECT_ROOT / "super_funsearch",
    Path.cwd() / "super_funsearch",
    Path.cwd() / "submission" / "super_funsearch_colab" / "super_funsearch",
    Path("/content/super_funsearch"),
]

SUPER_FUNSEARCH_DIR = None
for path in candidate_dirs:
    if (path / "bench_heuristic.py").exists() and (path / "bin_packing_utils.py").exists():
        SUPER_FUNSEARCH_DIR = path.resolve()
        break

if SUPER_FUNSEARCH_DIR is None:
    raise FileNotFoundError(
        "Could not find the submitted super_funsearch folder. "
        "Set REPO_URL or upload the folder containing super_funsearch/."
    )

sys.path.insert(0, str(SUPER_FUNSEARCH_DIR))
print("Using project code from:", SUPER_FUNSEARCH_DIR)

### Optional Dependency Installation

The report-table reproduction should run in a standard Colab runtime. If you want to run the full LLM pipeline, set `INSTALL_FULL_REQUIREMENTS = True` to install `requirements.txt`. This is disabled by default because packages such as `torch` and `sentence-transformers` are unnecessary for deterministic evaluation and can slow down setup.

In [ ]:
INSTALL_FULL_REQUIREMENTS = False

if INSTALL_FULL_REQUIREMENTS:
    req = SUPER_FUNSEARCH_DIR / "requirements.txt"
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)
    else:
        print("No requirements.txt found; skipping installation.")
else:
    print("Skipping full requirements installation for deterministic reproduction.")

## 2. Dataset and Evaluator

This cell imports the project evaluator and dataset utilities. `bin_packing_utils.py` contains the OR3 and Weibull 5k datasets and the L1 lower-bound computation. `bench_heuristic.py` contains the deterministic simulator and baseline implementations used in the report.

The reported columns are:

- `Avg.`: average number of bins used across instances.
- `Std.`: standard deviation across instances.
- `Min` / `Max`: best and worst bin counts across instances.
- `LB`: average L1 lower bound.
- `Gap`: `Avg. - LB`; lower is better.

In [ ]:
import numpy as np
import pandas as pd

import bench_heuristic
import bin_packing_utils

DATASET_NAME = "Weibull 5k"

print("Available datasets:", list(bin_packing_utils.datasets.keys()))
print("Selected dataset:", DATASET_NAME)
print("Number of Weibull 5k instances:", len(bin_packing_utils.datasets[DATASET_NAME]))

### Table Formatting Helpers

The project evaluator returns dictionaries. The helper functions below convert them into the compact table format used in the report.

In [ ]:
def stats_to_row(stats):
    """Convert a bench_heuristic stats dictionary into one report row."""
    return {
        "Method": stats["name"],
        "Avg.": stats["avg"],
        "Std.": stats["std"],
        "Min": stats["min"],
        "Max": stats["max"],
        "LB": stats["avg_l1"],
        "Gap": stats["avg_gap"],
    }

def format_report_table(rows):
    """Return a pandas DataFrame with rounded report-table values."""
    df = pd.DataFrame(rows)
    for col in ["Avg.", "Std.", "LB", "Gap"]:
        df[col] = df[col].map(lambda x: round(float(x), 2))
    return df

## 3. Classical Baselines

This cell reproduces the classical baseline rows in the report table by calling `bench_heuristic.bench_classical()`. FFD is an offline reference because it sorts all items before packing; the other baselines are online heuristics.

In [ ]:
baseline_names = [
    "First Fit",
    "Best Fit",
    "Worst Fit",
    "Next Fit",
    "First Fit Decreasing",
]

baseline_rows = []
for name in baseline_names:
    stats = bench_heuristic.bench_classical(name, dataset_name=DATASET_NAME)
    row = stats_to_row(stats)
    if name == "First Fit Decreasing":
        row["Method"] = "FFD (offline)"
    baseline_rows.append(row)

format_report_table(baseline_rows)

## 4. Super-FunSearch Architecture

This section documents the full pipeline without making LLM calls. The submitted code contains the complete A1/A2/A3/A4 system:

- **A1 Coder**: `implementation/eoh_operators.py` builds EoH-style thought/code prompts.
- **A2 Bug-Fixer**: `implementation/bug_fix_memory.py` stores deterministic runtime repair recipes.
- **A3 Algorithm-Guide**: `implementation/reevo_reflector.py` produces ReEvo-style short-term and long-term reflections.
- **A4 Search-Controller**: `implementation/search_controller.py` outputs constrained JSON search policies.
- **ProgramsDatabase**: `implementation/programs_database.py` stores islands, clusters, and parent candidates.
- **Sampler/Coordinator**: `implementation/sampler.py` routes generated code through evaluation, repair, reflection, and database insertion.

```mermaid
flowchart TD
  a4["A4 Search Controller"] -->|"policy: explore vs mutate"| sampler["Sampler"]
  db["ProgramsDatabase"] -->|"parents"| sampler
  sampler --> a1["A1 Coder"]
  a1 --> evaluator["Evaluator Sandbox"]
  evaluator --> dispatcher["Dispatcher"]
  dispatcher -->|"valid score"| a3["A3 Reflector"]
  dispatcher -->|"runtime crash"| a2["A2 Bug Fixer"]
  dispatcher -->|"store valid"| db
  a2 -->|"repaired code"| evaluator
```


### Architecture File Check

This cell verifies that the submitted package contains the core implementation files. It does not execute the LLM pipeline; it only checks that the code required by the optional full run is present.

In [ ]:
architecture_files = {
    "A1 Coder / EoH operators": "implementation/eoh_operators.py",
    "A2 Bug-Fixer memory": "implementation/bug_fix_memory.py",
    "A3 ReEvo reflector": "implementation/reevo_reflector.py",
    "A4 Search controller": "implementation/search_controller.py",
    "Programs database": "implementation/programs_database.py",
    "Sampler coordinator": "implementation/sampler.py",
    "Structure diagnostics": "implementation/structure_analysis.py",
    "Full run entry point": "run_super_funsearch.py",
}

for label, relative_path in architecture_files.items():
    path = SUPER_FUNSEARCH_DIR / relative_path
    print(f"{label:28s} -> {'OK' if path.exists() else 'MISSING'}  {relative_path}")

## 5. Saved Best Heuristic Reproduction

The report uses a saved v3 Weibull sample with search score `-2031.6`, corresponding to `Avg. Bins = 2031.60`. This cell loads `saved_samples/sample_000127_weibull_best.json` from the submitted package and evaluates it with the same deterministic evaluator used for the baselines.

If the sample file is unavailable, the notebook falls back to the function body printed in the report.

In [ ]:
sample_path = SUPER_FUNSEARCH_DIR / "saved_samples" / "sample_000127_weibull_best.json"

if sample_path.exists():
    priority_fn, raw_sample = bench_heuristic.load_priority_from_sample_json(sample_path)
    print("Loaded saved sample:", sample_path)
    print("Internal search score:", raw_sample.get("score"))
else:
    print("Saved sample not found; using embedded report heuristic.")
    def priority_fn(item: float, bins: np.ndarray) -> np.ndarray:
        """Fallback copy of the saved best Super-FunSearch heuristic."""
        multiples = np.round((bins - item) / item)
        return np.exp(-((bins - item - multiples * item) ** 2) / (bins * 0.05 + 0.5))

ours_stats = bench_heuristic.bench_priority_fn(
    priority_fn,
    dataset_name=DATASET_NAME,
    label="Ours (v3 sample 122)",
)
ours_row = stats_to_row(ours_stats)
ours_row

## 6. Final Report Table

This cell combines the classical baselines and the saved Super-FunSearch heuristic into the final table. This is the deterministic result that the project report cites.

In [ ]:
main_table = format_report_table(baseline_rows + [ours_row])
main_table

### Consistency Check

This small check prints the expected report values next to the observed values from this notebook. Minor formatting differences are acceptable, but the rounded values should match when the same dataset file is used.

In [ ]:
expected_report_values = {
    "Ours Avg.": 2031.60,
    "Ours Gap": 43.80,
    "Best Fit Avg.": 2067.00,
}

best_fit_row = next(row for row in baseline_rows if row["Method"] == "Best Fit")

print("Expected:", expected_report_values)
print("Observed Ours Avg.:", round(float(ours_row["Avg."]), 2))
print("Observed Ours Gap:", round(float(ours_row["Gap"]), 2))
print("Observed Best Fit Avg.:", round(float(best_fit_row["Avg."]), 2))

## 7. Prior-Work Context

FunSearch reports `0.68%` excess bins on Weibull 5k. EoH reports `0.80%` on Weibull 5k with capacity 100 and `0.78%` with capacity 500. These are prior-work excess-bin gaps, not direct `Avg. Bins` entries under the evaluator used in this notebook.

## 8. Optional Full Pipeline Run

The deterministic cells above reproduce the main report table without an API key. The full Super-FunSearch search can also be launched from this notebook, but it is optional because it:

- requires an OpenAI-compatible API key,
- calls LLMs for A1/A3/A4,
- can run for a long time,
- is stochastic and may not rediscover the exact saved sample.

Set `RUN_FULL_SEARCH = True` only if you want to run the full A1/A2/A3/A4 pipeline.

In [ ]:
RUN_FULL_SEARCH = False

if RUN_FULL_SEARCH:
    # Set your API key securely in Colab before running this cell, for example:
    # os.environ["OPENAI_API_KEY"] = "..."
    command = [
        sys.executable,
        "run_super_funsearch.py",
        "--provider", "openai",
        "--dataset", "weibull",
        "--max-samples", "30",
        "--num-islands", "4",
        "--samples-per-prompt", "1",
        "--reset-every-n-samples", "50",
        "--no-numba",
        "--search-controller",
        "--search-controller-horizon", "10",
        "--search-controller-min-events", "6",
    ]
    subprocess.run(command, cwd=str(SUPER_FUNSEARCH_DIR), check=True)
else:
    print("Full LLM search skipped. Set RUN_FULL_SEARCH = True and provide an API key to run it.")

## 9. Reproducibility Notes

- The baseline and saved-best cells are deterministic.
- The full evolutionary search is stochastic because candidate programs are sampled from an LLM.
- The internal Super-FunSearch score is negative average bin count. For example, score `-2031.6` corresponds to `Avg. Bins = 2031.60`.
- API keys should be supplied as Colab secrets or environment variables, never hard-coded in the notebook or repository.